In [ ]:
# =========================================================================
# 이 코드는 구글 코랩(Colab) GPU 환경에서 실행해야 합니다. (런타임 유형 -> T4 GPU 선택)
# =========================================================================

# 1. 최속 LLM 학습 라이브러리 (Unsloth) 및 필수 패키지 설치
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft
!pip install pandas llama-cpp-python

import torch
from unsloth import FastLanguageModel
from datasets import Dataset
import pandas as pd

# 2. 베이스 모델 및 설정값 세팅
max_seq_length = 2048
dtype = None # GPU 아키텍처에 맞게 자동 설정
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct", # 베이스 모델
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. LoRA 아키텍처 설정
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 4. CSV 데이터셋 로드 및 포맷팅
csv_path = "./qa_dataset.csv"
df = pd.read_csv(csv_path)

# AI가 대화의 맥락(ChatML 스타일)을 학습할 수 있도록 가이드 포맷 적용
df['text'] = df.apply(lambda row: f"<|im_start|>system\n당신은 노인 전문 병원의 친절한 간호사입니다.<|im_end|>\n<|im_start|>user\n{row['쉬운 질문']}<|im_end|>\n<|im_start|>assistant\n{row['답변']}<|im_end|>", axis=1)
dataset = Dataset.from_pandas(df[['text']])

# 5. 학습 트레이너 설정
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # 짧은 문장이 많으므로 패킹 취소
    args = TrainingArguments(
        per_device_train_batch_size = 2, #메모리 터지면 1로 축소
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # 전체 데이터양에 따라 조절 (예시: 60번 반복 학습)
        learning_rate = 2e-4, # 학습 속도율
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        save_strategy = "none",
    ),
)

# 학습 시작 버튼
print("의료 LLM 학습을 시작합니다.")
trainer_stats = trainer.train()

# 6. 노트북용 GGUF 파일로 변환하여 저장
print("백엔드 서버 이식용 GGUF 파일 변환 중...")
model.save_pretrained_gguf("model-unsloth", tokenizer, quantization_method = "q4_k_m")
print("GGUF 파일 생성이 완료되었습니다.")